# Review Orchestrator (`fastgit` + `ghapi` + `fastcdp`)

> Idempotent review-app and pull-request orchestrator built on `fastgit`, `ghapi`, and `fastcdp`.

**Guardrails:**
1. **Non-destructive branching**: `FastGitAdapter.create_branch` uses `git commit-tree` + `git update-ref` from `origin/<base_branch>` (or local `<base_branch>`) so uncommitted edits and untracked files in the operator's working tree are never touched, while creating 1 PR-ready commit ahead of base.
2. **Idempotent orchestration**: `ReviewOrchestrator.assign_tickets` inspects `ticket.json` first and reuses existing branches, open PRs, and running review apps instead of duplicating resources.
3. **Cleanup as a first-class action**: `complete_review(action="merge"|"close")` merges/closes the PR, tears down the Solveit review app, deletes the branch, and updates every linked `ticket.json`.

In [ ]:
#| default_exp orchestrator

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import secrets, subprocess
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any
from fastcore.utils import *

from craft_support.drive import TicketStore
from craft_support.models import ReviewManifest, Ticket
from craft_support.solveit_cdp import ReviewAppInfo, ReviewAppProvisioner

## Git Adapter (`fastgit`)

In [ ]:
#| export
@dataclass
class PullRequestInfo:
    "Summary of a GitHub pull request linked to one or more support tickets."
    number: int
    url: str
    branch: str
    base_branch: str
    state: str = "open"
    merged: bool = False

class GitAdapter(ABC):
    "Abstract async interface for branch management."
    @abstractmethod
    async def branch_exists(self, branch: str) -> bool: ...
    @abstractmethod
    async def create_branch(self, branch: str, *, base_branch: str = "main") -> str: ...
    @abstractmethod
    async def push_branch(self, branch: str, *, remote: str = "origin") -> None: ...
    @abstractmethod
    async def delete_branch(self, branch: str, *, remote: str | None = "origin") -> None: ...

class FastGitAdapter(GitAdapter, BasicRepr):
    "Local git operations powered by `AnswerDotAI/fastgit`."
    def __init__(
        self,
        repo_path: Path | str, # Local git repository path
        *,
        push_enabled: bool = True, # Push branches and fetch remote base refs when True
    ) -> None:
        from fastgit import Git
        self.repo_path = Path(repo_path)
        self.push_enabled = push_enabled
        self.git = Git(str(self.repo_path), sync=False, raise_exc=True)

    async def _ref_exists(self, ref: str) -> bool:
        try: return bool((await self.git.rev_parse("--verify", ref) or "").strip())
        except Exception: return False

    async def resolve_base_ref(self, base_branch: str, *, remote: str = "origin") -> str:
        "Resolve `base_branch`, preferring `origin/<base_branch>` when pushing is enabled."
        if self.push_enabled:
            try: await self.git.fetch(remote, base_branch)
            except Exception: pass
            remote_ref = f"{remote}/{base_branch}"
            if await self._ref_exists(remote_ref): return remote_ref
        if await self._ref_exists(base_branch): return base_branch
        raise RuntimeError(f"Base branch {base_branch!r} not found locally or on {remote}.")

    async def branch_exists(self, branch: str) -> bool:
        try: return bool((await self.git.branch("--list", branch) or "").strip())
        except Exception: return False

    async def create_branch(self, branch: str, *, base_branch: str = "main", ensure_commit: bool = True) -> str:
        if await self.branch_exists(branch): return branch
        base_ref = await self.resolve_base_ref(base_branch)
        # Non-destructive commit-tree + update-ref creates a PR-ready commit without touching the working tree.
        if ensure_commit:
            try:
                tree_sha = (await self.git.rev_parse(f"{base_ref}^{{tree}}")).strip()
                parent_sha = (await self.git.rev_parse(base_ref)).strip()
                commit_sha = (await self.git.commit_tree(
                    tree_sha, "-p", parent_sha, "-m", f"chore(support): initialize review branch {branch}")).strip()
                await self.git.update_ref(f"refs/heads/{branch}", commit_sha)
                return branch
            except Exception: pass
        await self.git.branch(branch, base_ref)
        return branch

    async def push_branch(self, branch: str, *, remote: str = "origin") -> None:
        if self.push_enabled: await self.git.push("-u", remote, branch)

    async def delete_branch(self, branch: str, *, remote: str | None = "origin") -> None:
        if await self.branch_exists(branch): await self.git.branch("-D", branch)
        if remote and self.push_enabled:
            try: await self.git.push(remote, "--delete", branch)
            except Exception: pass

In [ ]:
show_doc(FastGitAdapter)

In [ ]:
import asyncio, tempfile
from unittest.mock import patch as mock_patch
from fastcore.test import test_eq
from craft_support.drive import MemoryTicketStore
from craft_support.solveit_cdp import MemoryReviewAppProvisioner

In [ ]:
def _init_temp_git_repo(root: Path) -> Path:
    repo = root/"repo"
    repo.mkdir()
    subprocess.run(["git", "init", "-b", "main"], cwd=repo, check=True, capture_output=True)
    subprocess.run(["git", "config", "user.email", "test@example.com"], cwd=repo, check=True)
    subprocess.run(["git", "config", "user.name", "Test User"], cwd=repo, check=True)
    (repo/"README.md").write_text("# test\n", encoding="utf-8")
    subprocess.run(["git", "add", "README.md"], cwd=repo, check=True)
    subprocess.run(["git", "commit", "-m", "initial"], cwd=repo, check=True, capture_output=True)
    return repo

async def _test_fastgit_non_destructive():
    with tempfile.TemporaryDirectory() as td:
        repo = _init_temp_git_repo(Path(td))
        dirty, untracked = repo/"README.md", repo/"scratch.txt"
        dirty.write_text("# dirty uncommitted edit\n", encoding="utf-8")
        untracked.write_text("keep me\n", encoding="utf-8")

        git = FastGitAdapter(repo, push_enabled=False)
        await git.create_branch("support/test-dirty", base_branch="main")

        test_eq(dirty.read_text(encoding="utf-8"), "# dirty uncommitted edit\n")
        test_eq(untracked.read_text(encoding="utf-8"), "keep me\n")
        rev_count = subprocess.run(['git', 'rev-list', '--count', 'main..support/test-dirty'], cwd=repo, check=True, capture_output=True,
            text=True).stdout.strip()
        test_eq(rev_count, "1")

await _test_fastgit_non_destructive()

## GitHub Adapter (`ghapi`) & Token Resolution

`_resolve_github_token` checks an explicit argument first, then `GITHUB_TOKEN` / `GH_TOKEN`, and finally falls back to `gh auth token` so operators already logged into the GitHub CLI do not need to export extra environment variables.

In [ ]:
#| export
def _resolve_github_token(token: str | None = None) -> str | None:
    "Resolve a GitHub token from explicit arg, env (`GITHUB_TOKEN`/`GH_TOKEN`), or `gh auth token`."
    if token: return token
    if env_tok := os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN"): return env_tok.strip()
    try:
        res = subprocess.run(["gh", "auth", "token"], capture_output=True, text=True, check=False, timeout=10)
        if res.returncode == 0 and res.stdout.strip(): return res.stdout.strip()
    except Exception: pass
    return None

def _pr_info(pr, *, branch: str | None = None, base_branch: str | None = None, state: str | None = None, merged: bool = False) -> PullRequestInfo:
    return PullRequestInfo(number=int(pr.number), url=str(pr.html_url), branch=branch or str(pr.head.ref),
        base_branch=base_branch or str(pr.base.ref), state=state or str(pr.state), merged=merged)

class GhAdapter(ABC):
    "Abstract async interface for GitHub pull-request lifecycle operations."
    @abstractmethod
    async def find_open_pr(self, *, head_branch: str) -> PullRequestInfo | None: ...
    @abstractmethod
    async def create_pr(self, *, title: str, body: str, head: str, base: str) -> PullRequestInfo: ...
    @abstractmethod
    async def comment_pr(self, pr_number: int, body: str) -> None: ...
    @abstractmethod
    async def merge_pr(self, pr_number: int, *, merge_method: str = "squash") -> PullRequestInfo: ...
    @abstractmethod
    async def close_pr(self, pr_number: int) -> PullRequestInfo: ...

class GhApiAdapter(GhAdapter, BasicRepr):
    "GitHub pull-request operations powered by `fastai/ghapi`."
    def __init__(
        self,
        owner: str, # GitHub repository owner
        repo: str, # GitHub repository name
        *,
        token: str | None = None, # Explicit GitHub token (falls back to env or `gh auth token`)
    ) -> None:
        from ghapi.all import GhApi
        store_attr("owner,repo")
        self.api = GhApi(owner=owner, repo=repo, token=_resolve_github_token(token))

    async def find_open_pr(self, *, head_branch: str) -> PullRequestInfo | None:
        pr = first(await self.api.pulls.list(state="open", head=f"{self.owner}:{head_branch}"))
        return _pr_info(pr) if pr else None

    async def create_pr(self, *, title: str, body: str, head: str, base: str) -> PullRequestInfo:
        if existing := await self.find_open_pr(head_branch=head): return existing
        pr = await self.api.pulls.create(title=title, body=body, head=head, base=base)
        return _pr_info(pr, branch=head, base_branch=base)

    async def comment_pr(self, pr_number: int, body: str) -> None: await self.api.issues.create_comment(issue_number=pr_number, body=body)

    async def merge_pr(self, pr_number: int, *, merge_method: str = "squash") -> PullRequestInfo:
        await self.api.pulls.merge(pull_number=pr_number, merge_method=merge_method)
        return _pr_info(await self.api.pulls.get(pull_number=pr_number), state="closed", merged=True)

    async def close_pr(self, pr_number: int) -> PullRequestInfo:
        return _pr_info(await self.api.pulls.update(pull_number=pr_number, state="closed"), state="closed", merged=False)

In [ ]:
show_doc(GhApiAdapter)

In [ ]:
test_eq(_resolve_github_token("explicit-tok"), "explicit-tok")
with mock_patch.dict(os.environ, dict(GITHUB_TOKEN="env-tok"), clear=False): test_eq(_resolve_github_token(), "env-tok")

class _FakeProc:
    returncode, stdout = 0, "gh-cli-tok\n"

with mock_patch.dict(os.environ, {}, clear=True), mock_patch.object(subprocess, "run", return_value=_FakeProc()):
    test_eq(_resolve_github_token(), "gh-cli-tok")

## `ReviewOrchestrator`

Coordinates the four steps of ticket assignment and the inverse cleanup flow on merge/close. Supports stacking a new review branch on top of an existing review branch via `base_branch`.

In [ ]:
#| export
@dataclass
class AssignmentResult:
    "Outcome of assigning one or more tickets to a branch, PR, and Solveit review app."
    group_id: str
    branch: str
    base_branch: str
    pr: PullRequestInfo
    review_app: ReviewAppInfo
    tickets: list[Ticket]
    manifest: ReviewManifest
    reused: bool = False

def _default_branch_name(tickets: list[Ticket]) -> str:
    ids = "-".join(t.id.lower() for t in tickets[:2])
    slug = re.sub(r"[^a-z0-9-]+", "-", tickets[0].title.lower()).strip("-")[:24]
    return f"support/{ids}-{slug}".rstrip("-")

def _render_pr_body(tickets: list[Ticket], *, base_branch: str, group_id: str) -> str:
    lines = [f"## Support Review Bundle (`{group_id}`)", "", f"Base branch: `{base_branch}`", "", "### Linked Tickets (Google Drive)"]
    for t in tickets:
        folder = t.drive.folder_url or "(Drive folder)"
        dom = f" | [dom.html]({t.drive.dom_url})" if t.drive.dom_url else ""
        shot = f" | [screenshot.png]({t.drive.screenshot_url})" if t.drive.screenshot_url else ""
        lines.append(f"- **{t.id}** — {t.title} ([Drive folder]({folder}){dom}{shot})")
        if t.description: lines.append(f"  - {t.description.splitlines()[0][:180]}")
        for note in t.triage_notes: lines.append(f"  - *Triage ({note.author}):* {note.text}")
    return "\n".join(lines)

class ReviewOrchestrator(BasicRepr):
    "Orchestrates git branches (`fastgit`), GitHub PRs (`ghapi`), and Solveit review apps (`fastcdp`)."
    def __init__(
        self,
        *,
        store: TicketStore, # Ticket storage backend
        git: GitAdapter, # Git branch adapter
        gh: GhAdapter, # GitHub pull-request adapter
        provisioner: ReviewAppProvisioner, # Solveit review-app provisioner
        assets_dir: Path | None = None, # Optional compiled CRAFT `dist/` directory
    ) -> None:
        store_attr()

### Ticket Assignment (`assign_tickets`)

Inspects each ticket's stored links first so retries reuse the existing branch, PR, and Solveit review app rather than creating duplicates.

In [ ]:
#| export
@patch
async def assign_tickets(
    self: ReviewOrchestrator,
    ticket_ids: list[str], # One or more ticket IDs to group into one review app and PR
    *,
    base_branch: str = "main", # Target base branch (or parent review branch for stacking)
    existing_branch: str | None = None, # Attach to an existing branch instead of generating one
    branch_name: str | None = None, # Custom branch name override
    title: str | None = None, # Custom PR title override
) -> AssignmentResult:
    "Assign ticket(s) idempotently to a branch, PR, and Solveit review app."
    if not ticket_ids: raise ValueError("At least one ticket_id is required")
    tickets = [await self.store.get_ticket(tid) for tid in ticket_ids]

    existing_branches = {t.branch for t in tickets if t.branch}
    existing_prs = {t.pr_number for t in tickets if t.pr_number}
    existing_apps = {t.review_app_id for t in tickets if t.review_app_id}
    existing_groups = {t.group_id for t in tickets if t.group_id}

    if len(existing_branches) > 1 or len(existing_prs) > 1 or len(existing_apps) > 1:
        raise ValueError(
            "Selected tickets are already assigned to conflicting review apps/branches. "
            "Assign them separately or clear stale links first.")

    reused = bool(existing_branches and existing_prs and existing_apps)
    group_id = first(existing_groups) or f"GRP-{secrets.token_hex(3).upper()}"
    branch = existing_branch or first(existing_branches) or branch_name or _default_branch_name(tickets)
    effective_base = first(t.base_branch for t in tickets if t.base_branch) or base_branch

    if not await self.git.branch_exists(branch): await self.git.create_branch(branch, base_branch=effective_base)
    await self.git.push_branch(branch)

    pr_number, pr_url = first(existing_prs), first(t.pr_url for t in tickets if t.pr_url)
    if pr_number and pr_url: pr = PullRequestInfo(number=pr_number, url=pr_url, branch=branch, base_branch=effective_base)
    else:
        pr = await self.gh.find_open_pr(head_branch=branch)
        if pr is None:
            suffix = f" +{len(tickets) - 1}" if len(tickets) > 1 else ""
            pr_title = title or f"[{tickets[0].id}{suffix}] {tickets[0].title}"
            pr = await self.gh.create_pr(title=pr_title, body=_render_pr_body(tickets, base_branch=effective_base, group_id=group_id),
                head=branch, base=effective_base)

    manifest = ReviewManifest.from_tickets(
        tickets, group_id=group_id, branch=branch, base_branch=effective_base, pr_number=pr.number, pr_url=pr.url)
    existing_app_id = first(existing_apps)
    review_app = await self.provisioner.inspect(existing_app_id) if existing_app_id else None
    newly_provisioned = review_app is None
    if review_app is None:
        review_app = await self.provisioner.provision(
            branch=branch, manifest=manifest, instance_name=existing_app_id, assets_dir=self.assets_dir)

    manifest.review_app_id, manifest.review_app_url = review_app.review_app_id, review_app.url
    if newly_provisioned:
        await self.gh.comment_pr(pr.number, (
            f"Solveit review app provisioned for branch `{branch}`:\n"
            f"- **Review App:** {review_app.url} (`{review_app.review_app_id}`)\n"
            f"- **Tickets:** {', '.join(t.id for t in tickets)}\n"
            f"- **Manifest:** `{review_app.manifest_path or 'review_manifest.json'}`"))

    updated = []
    for t in tickets:
        t.group_id, t.branch, t.base_branch = group_id, branch, effective_base
        t.pr_number, t.pr_url = pr.number, pr.url
        t.review_app_id, t.review_app_url = review_app.review_app_id, review_app.url
        if t.status in ("new", "triaged"): t.set_status("in_review")
        updated.append(await self.store.update_ticket(t))

    return AssignmentResult(group_id=group_id, branch=branch, base_branch=effective_base, pr=pr, review_app=review_app, tickets=updated,
        manifest=manifest, reused=reused and (not newly_provisioned))

### Review Completion & Cleanup (`complete_review`, `teardown_review_app`)

In [ ]:
#| export
@patch
async def complete_review(
    self: ReviewOrchestrator,
    ticket_ids: list[str], # Ticket IDs whose PR and review app should be finalized
    *,
    action: str = "merge", # Either `"merge"` or `"close"`
    merge_method: str = "squash", # GitHub merge method (`merge`, `squash`, or `rebase`)
    delete_branch: bool = True, # Delete local and remote branch after completion
) -> dict[str, Any]:
    "Merge or close the PR, tear down the Solveit review app, and update `ticket.json` in Drive."
    if action not in ("merge", "close"): raise ValueError(f"action must be 'merge' or 'close', got {action!r}")
    if not ticket_ids: raise ValueError("At least one ticket_id is required")

    tickets = [await self.store.get_ticket(tid) for tid in ticket_ids]
    pr_numbers = {t.pr_number for t in tickets if t.pr_number}
    review_app_ids = {t.review_app_id for t in tickets if t.review_app_id}
    branches = {t.branch for t in tickets if t.branch}

    pr_result = None
    for pr_num in pr_numbers:
        pr_result = await (self.gh.merge_pr(pr_num, merge_method=merge_method) if action == "merge" else self.gh.close_pr(pr_num))

    torndown = [app_id for app_id in review_app_ids if await self.provisioner.teardown(app_id)]
    if delete_branch:
        for br in branches: await self.git.delete_branch(br)

    new_status = "merged" if action == "merge" else "closed"
    updated = []
    for t in tickets:
        t.set_status(new_status)  # type: ignore[arg-type]
        updated.append(await self.store.update_ticket(t))

    return dict(action=action, status=new_status, pr=pr_result, torndown_apps=torndown,
        deleted_branches=sorted(branches) if delete_branch else [], tickets=updated)

@patch
async def teardown_review_app(self: ReviewOrchestrator, review_app_id: str) -> bool:
    "Explicit standalone cleanup for a Solveit review app."
    return await self.provisioner.teardown(review_app_id)

In [ ]:
show_doc(ReviewOrchestrator)

In [ ]:
#| hide
class _FakeGhAdapter(GhAdapter):
    def __init__(self): self.prs, self.comments, self._next_num, self.create_calls = {}, {}, 100, 0

    async def find_open_pr(self, *, head_branch: str):
        return first(pr for pr in self.prs.values() if pr.branch == head_branch and pr.state == "open")

    async def create_pr(self, *, title: str, body: str, head: str, base: str):
        self.create_calls += 1
        self._next_num += 1
        pr = PullRequestInfo(number=self._next_num, url=f"https://github.com/abiancar/solve-it-crafts/pull/{self._next_num}", branch=head, base_branch=base)
        self.prs[pr.number] = pr
        self.comments[pr.number] = [body]
        return pr

    async def comment_pr(self, pr_number: int, body: str): self.comments.setdefault(pr_number, []).append(body)

    async def merge_pr(self, pr_number: int, *, merge_method: str = "squash"):
        pr = self.prs[pr_number]
        pr.state, pr.merged = "closed", True
        return pr

    async def close_pr(self, pr_number: int):
        pr = self.prs[pr_number]
        pr.state, pr.merged = "closed", False
        return pr

async def _test_orchestrator_lifecycle():
    with tempfile.TemporaryDirectory() as td:
        repo = _init_temp_git_repo(Path(td))
        store, git, gh, prov = MemoryTicketStore(), FastGitAdapter(repo, push_enabled=False), _FakeGhAdapter(), MemoryReviewAppProvisioner()
        orch = ReviewOrchestrator(store=store, git=git, gh=gh, provisioner=prov)

        t1, _ = await store.create_ticket(title="Fix focus ring", description="Ring offset off by 2px")
        t2, _ = await store.create_ticket(title="Fix code wrap", description="Long lines overflow")

        res1 = await orch.assign_tickets([t1.id, t2.id])
        test_eq((res1.reused, res1.pr.number, len(res1.manifest.tickets)), (False, 101, 2))

        res_retry = await orch.assign_tickets([t1.id, t2.id])
        test_eq((res_retry.reused, gh.create_calls, prov.provision_calls), (True, 1, 1))

        t3, _ = await store.create_ticket(title="Stacked theme tweak", description="Depends on focus ring")
        res_stack = await orch.assign_tickets([t3.id], base_branch=res1.branch)
        test_eq((res_stack.base_branch, res_stack.pr.base_branch), (res1.branch, res1.branch))

        merged = await orch.complete_review([t1.id, t2.id], action="merge")
        test_eq(merged["status"], "merged")
        assert res1.review_app.review_app_id not in prov.instances and not await git.branch_exists(res1.branch)

        closed = await orch.complete_review([t3.id], action="close")
        test_eq(closed["status"], "closed")
        assert res_stack.review_app.review_app_id not in prov.instances

await _test_orchestrator_lifecycle()